# 📈 Supplementary EDA: Hillstrom Uplift, Criteo v2 & Tiki Product Catalog

> **Sprint 1 — Task 1.3+**  
> Analyze Randomized Controlled Trial (RCT) uplift datasets for the **Optimal Discount / Uplift Model (`CausalML`)** and profile the **Tiki Vietnamese Product Catalog** used to ground LLM recovery messages.


In [ ]:
import glob
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
HILLSTROM_DIR = '../data/raw/hillstrom/'
CRITEO_DIR = '../data/raw/criteo_uplift/'
TIKI_DIR = '../data/raw/tiki_shopee_catalog/'


## 1. 🧪 Hillstrom Email Marketing — RCT Balance & Average Treatment Effect (ATE)

In [ ]:
hill_files = sorted(glob.glob(f'{HILLSTROM_DIR}*.csv'))
df_hill = pd.read_csv(hill_files[0])
print(f'Hillstrom dataset shape: {df_hill.shape}')
display(df_hill.head())

# Compute Average Treatment Effect summary across RCT arms
ate_summary = df_hill.groupby('segment').agg(
    customers=('visit', 'count'),
    visit_rate=('visit', 'mean'),
    conversion_rate=('conversion', 'mean'),
    avg_spend=('spend', 'mean'),
).round(4)
display(ate_summary)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, title in zip(
    axes,
    ['visit_rate', 'conversion_rate', 'avg_spend'],
    ['Visit Rate by Arm', 'Conversion Rate by Arm', 'Mean Spend ($) by Arm'],
):
    sns.barplot(x=ate_summary.index, y=ate_summary[col], ax=ax, palette='viridis')
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()


## 2. 🎯 Empirical Conversion Uplift by Customer Recency

In [ ]:
df_hill['treatment'] = (df_hill['segment'] != 'No E-Mail').astype(int)
uplift_by_rec = df_hill.pivot_table(
    index='recency', columns='treatment', values='conversion', aggfunc='mean'
)
uplift_by_rec['empirical_uplift'] = uplift_by_rec[1] - uplift_by_rec[0]

plt.figure(figsize=(12, 5))
plt.bar(uplift_by_rec.index, uplift_by_rec['empirical_uplift'] * 100, color='#27ae60')
plt.axhline(0, color='black', linestyle='--')
plt.xlabel('Recency (Months Since Last Purchase)')
plt.ylabel('Incremental Conversion Lift (% points)')
plt.title('Empirical Conversion Uplift by Customer Recency', fontweight='bold')
plt.show()


## 3. 🛍️ Tiki Vietnamese Fashion Catalog — Grounding Dataset Profile

In [ ]:
tiki_files = sorted(glob.glob(f'{TIKI_DIR}*.csv'))
df_tiki = pd.concat([pd.read_csv(f) for f in tiki_files], ignore_index=True)
df_tiki['discount_pct'] = (
    (df_tiki['original_price'] - df_tiki['price']) / df_tiki['original_price'].clip(lower=1) * 100
).clip(0, 100)
print(f'Total Vietnamese catalog items loaded: {len(df_tiki):,}')
display(df_tiki[['id', 'name', 'brand', 'price', 'original_price', 'discount_pct', 'rating_average']].head(10))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_tiki['price'].clip(upper=df_tiki['price'].quantile(0.98)) / 1000, bins=40, color='#3498db')
axes[0].set_title('Selling Price Distribution (x1,000 VND)', fontweight='bold')
axes[1].hist(df_tiki['discount_pct'], bins=30, color='#e67e22')
axes[1].set_title('Listed Discount Percentage Distribution', fontweight='bold')
plt.tight_layout()
plt.show()
